In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_curve, average_precision_score
from sklearn.ensemble import RandomForestClassifier

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

class DefaultRateAnalysis:
    def __init__(self, random_state=42):
        self.random_state = random_state
        
    def demonstrate_impact_of_different_rates(self, n_samples=10000):
        """
        Demonstrate the impact of different default rates in train/test sets
        """
        # Generate base dataset
        np.random.seed(self.random_state)
        df = self.generate_credit_data(n_samples)
        
        # Create splits with different methods
        splits = {
            'stratified': self.create_stratified_split(df),
            'random': self.create_random_split(df),
            'biased': self.create_biased_split(df)
        }
        
        # Analyze each split
        results = {}
        for split_name, (X_train, X_test, y_train, y_test) in splits.items():
            results[split_name] = self.analyze_split(
                X_train, X_test, y_train, y_test, split_name
            )

        return results
    
    def generate_credit_data(self, n_samples):
        """Generate synthetic credit scoring data"""
        df = pd.DataFrame({
            'income': np.random.normal(50000, 20000, n_samples),
            'debt_ratio': np.random.uniform(0.1, 0.6, n_samples),
            'credit_score': np.random.normal(700, 50, n_samples),
            'default': np.random.choice([0, 1], n_samples, p=[0.95, 0.05])
        })
        
        # Add realistic correlations
        df.loc[df['credit_score'] < 650, 'default'] = \
            np.random.choice([0, 1], sum(df['credit_score'] < 650), p=[0.8, 0.2])
        return df
    
    def create_stratified_split(self, df):
        """Create stratified train/test split"""
        return train_test_split(
            df.drop('default', axis=1),
            df['default'],
            test_size=0.2,
            stratify=df['default'],
            random_state=self.random_state
        )
    
    def create_random_split(self, df):
        """Create random (non-stratified) split"""
        return train_test_split(
            df.drop('default', axis=1),
            df['default'],
            test_size=0.2,
            stratify=None,
            random_state=self.random_state
        )
    
    def create_biased_split(self, df):
        """Create intentionally biased split"""
        # Sort by credit_score to create biased splits
        df_sorted = df.sort_values('credit_score')
        split_idx = int(len(df) * 0.8)
        
        X_train = df_sorted.iloc[:split_idx].drop('default', axis=1)
        X_test = df_sorted.iloc[split_idx:].drop('default', axis=1)
        y_train = df_sorted.iloc[:split_idx]['default']
        y_test = df_sorted.iloc[split_idx:]['default']
        
        return X_train, X_test, y_train, y_test
    
    def analyze_split(self, X_train, X_test, y_train, y_test, split_name):
        """Analyze the impact of split on model performance"""
        # Train model
        model = RandomForestClassifier(random_state=self.random_state)
        model.fit(X_train, y_train)
        
        # Get predictions
        y_pred_train = model.predict_proba(X_train)[:, 1]
        y_pred_test = model.predict_proba(X_test)[:, 1]
        
        # Calculate metrics
        results = {
            'default_rates': {
                'train': y_train.mean(),
                'test': y_test.mean(),
                'difference': abs(y_train.mean() - y_test.mean())
            },
            'performance': {
                'train_auc': roc_auc_score(y_train, y_pred_train),
                'test_auc': roc_auc_score(y_test, y_pred_test),
                'auc_difference': abs(
                    roc_auc_score(y_train, y_pred_train) - 
                    roc_auc_score(y_test, y_pred_test)
                )
            },
            'calibration': {
                'train_avg_pred': y_pred_train.mean(),
                'test_avg_pred': y_pred_test.mean(),
                'prediction_bias': y_pred_test.mean() - y_test.mean()
            }
        }
        
        return results

def run_demonstration():
    """Run demonstration and display results"""
    analyzer = DefaultRateAnalysis()
    results = analyzer.demonstrate_impact_of_different_rates()
    print("number of results is ", len(results))
    # Print results
    print("\nImpact of Different Default Rates in Train/Test Sets")
    print("=" * 50)
    
    for split_name, metrics in results.items():
        print(f"\n{split_name.capitalize()} Split:")
        print("-" * 30)
        
        # Default rates
        dr = metrics['default_rates']
        print(f"Default Rates:")
        print(f"  Train: {dr['train']:.3f}")
        print(f"  Test:  {dr['test']:.3f}")
        print(f"  Difference: {dr['difference']:.3f}")
        
        # Performance
        perf = metrics['performance']
        print(f"\nModel Performance:")
        print(f"  Train AUC: {perf['train_auc']:.3f}")
        print(f"  Test AUC:  {perf['test_auc']:.3f}")
        print(f"  AUC Difference: {perf['auc_difference']:.3f}")
        
        # Calibration
        cal = metrics['calibration']
        print(f"\nModel Calibration:")
        print(f"  Prediction Bias: {cal['prediction_bias']:.3f}")
    
    return results

# Run the demonstration
results = run_demonstration()

number of results is  3

Impact of Different Default Rates in Train/Test Sets

Stratified Split:
------------------------------
Default Rates:
  Train: 0.073
  Test:  0.073
  Difference: 0.000

Model Performance:
  Train AUC: 1.000
  Test AUC:  0.587
  AUC Difference: 0.413

Model Calibration:
  Prediction Bias: 0.003

Random Split:
------------------------------
Default Rates:
  Train: 0.073
  Test:  0.074
  Difference: 0.001

Model Performance:
  Train AUC: 1.000
  Test AUC:  0.594
  AUC Difference: 0.406

Model Calibration:
  Prediction Bias: 0.001

Biased Split:
------------------------------
Default Rates:
  Train: 0.079
  Test:  0.053
  Difference: 0.025

Model Performance:
  Train AUC: 1.000
  Test AUC:  0.548
  AUC Difference: 0.452

Model Calibration:
  Prediction Bias: 0.320


In [9]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

class ResultDisplay:
    def __init__(self):
        self.style_properties = """
        <style>
            .styled-table {
                border-collapse: collapse;
                margin: 25px 0;
                font-size: 0.9em;
                font-family: sans-serif;
                min-width: 400px;
                box-shadow: 0 0 20px rgba(0, 0, 0, 0.15);
            }
            .styled-table thead tr {
                background-color: #009879;
                color: #ffffff;
                text-align: left;
            }
            .styled-table th,
            .styled-table td {
                padding: 12px 15px;
            }
            .styled-table tbody tr {
                border-bottom: 1px solid #dddddd;
            }
            .styled-table tbody tr:nth-of-type(even) {
                background-color: #f3f3f3;
            }
            .section-header {
                background-color: #4CAF50;
                color: white;
                padding: 10px;
                margin-top: 20px;
                margin-bottom: 10px;
            }
        </style>
        """
    
    def display_section_header(self, title):
        """Display a formatted section header"""
        display(HTML(f"""
        {self.style_properties}
        <div class="section-header">{title}</div>
        """))

    def display_dataframe(self, df, title=None):
        """Display a formatted DataFrame"""
        if title:
            self.display_section_header(title)
        
        styled_df = df.style.set_properties(**{
            'background-color': '#f4f4f4',
            'padding': '10px',
            'border': '1px solid #ddd'
        })
        display(styled_df)

    def display_dict_as_table(self, data, title=None):
        """Convert dictionary to DataFrame and display"""
        if isinstance(data, dict):
            if all(isinstance(v, dict) for v in data.values()):
                # Nested dictionary
                df = pd.DataFrame(data).T
            else:
                # Single level dictionary
                df = pd.DataFrame(list(data.items()), columns=['Key', 'Value'])
        else:
            df = pd.DataFrame(data)
        
        self.display_dataframe(df, title)

def analyze_and_display_results(results):
    """
    Analyze and display comprehensive results
    """
    display_helper = ResultDisplay()
    
    # 1. Default Rate Analysis
    default_rates = {split: {
        'Train Default Rate': metrics['default_rates']['train'],
        'Test Default Rate': metrics['default_rates']['test'],
        'Rate Difference': metrics['default_rates']['difference']
    } for split, metrics in results.items()}
    
    display_helper.display_dict_as_table(
        pd.DataFrame(default_rates).round(4),
        "Default Rate Analysis"
    )
    
    # 2. Model Performance Metrics
    performance_metrics = {split: {
        'Train AUC': metrics['performance']['train_auc'],
        'Test AUC': metrics['performance']['test_auc'],
        'AUC Difference': metrics['performance']['auc_difference']
    } for split, metrics in results.items()}
    
    display_helper.display_dict_as_table(
        pd.DataFrame(performance_metrics).round(4),
        "Model Performance Metrics"
    )
    
    # 3. Calibration Analysis
    calibration_metrics = {split: {
        'Train Avg Prediction': metrics['calibration']['train_avg_pred'],
        'Test Avg Prediction': metrics['calibration']['test_avg_pred'],
        'Prediction Bias': metrics['calibration']['prediction_bias']
    } for split, metrics in results.items()}
    
    display_helper.display_dict_as_table(
        pd.DataFrame(calibration_metrics).round(4),
        "Calibration Analysis"
    )
    
    # 4. Statistical Tests
    statistical_tests = calculate_statistical_tests(results)
    display_helper.display_dict_as_table(
        pd.DataFrame(statistical_tests).round(4),
        "Statistical Tests"
    )
    
    # 5. Business Impact Analysis
    business_impact = calculate_business_impact(results)
    display_helper.display_dict_as_table(
        pd.DataFrame(business_impact).round(4),
        "Business Impact Analysis"
    )

def calculate_statistical_tests(results):
    """Calculate additional statistical tests"""
    from scipy import stats
    
    statistical_tests = {}
    for split, metrics in results.items():
        train_rate = metrics['default_rates']['train']
        test_rate = metrics['default_rates']['test']
        
        # Chi-square test for independence
        chi2, p_value = stats.chi2_contingency([[
            train_rate * 100, (1-train_rate) * 100
        ], [
            test_rate * 100, (1-test_rate) * 100
        ]])[0:2]
        
        statistical_tests[split] = {
            'Chi-Square Statistic': chi2,
            'P-Value': p_value,
            'Is Significant': p_value < 0.05
        }
    
    return statistical_tests

def calculate_business_impact(results):
    """Calculate business impact metrics"""
    business_impact = {}
    for split, metrics in results.items():
        rate_diff = metrics['default_rates']['difference']
        
        business_impact[split] = {
            'Expected Loss Impact (%)': rate_diff * 100,
            'Portfolio Risk Adjustment': abs(1 - (1 / (1 + rate_diff))),
            'Threshold Adjustment Needed': abs(0.5 - (0.5 * (1 + rate_diff)))
        }
    
    return business_impact

# Modified run_demonstration function
def run_comprehensive_demonstration():
    """Run demonstration with comprehensive results display"""
    analyzer = DefaultRateAnalysis()
    results = analyzer.demonstrate_impact_of_different_rates()
    
    # Display comprehensive results
    analyze_and_display_results(results)
    
    return results

# Run the comprehensive demonstration
results = run_comprehensive_demonstration()

# Additional visualization
def plot_comprehensive_results(results):
    """Create comprehensive visualizations of results"""
    plt.style.use('seaborn')
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Comprehensive Analysis of Different Splitting Methods')
    
    # 1. Default Rates Comparison
    default_rates = pd.DataFrame({
        split: [metrics['default_rates']['train'], 
               metrics['default_rates']['test']]
        for split, metrics in results.items()
    }, index=['Train', 'Test'])
    
    default_rates.plot(kind='bar', ax=axes[0,0])
    axes[0,0].set_title('Default Rates by Split Method')
    axes[0,0].set_ylabel('Default Rate')
    
    # 2. AUC Comparison
    auc_scores = pd.DataFrame({
        split: [metrics['performance']['train_auc'],
               metrics['performance']['test_auc']]
        for split, metrics in results.items()
    }, index=['Train', 'Test'])
    
    auc_scores.plot(kind='bar', ax=axes[0,1])
    axes[0,1].set_title('AUC Scores by Split Method')
    axes[0,1].set_ylabel('AUC Score')
    
    # 3. Calibration Plot
    calibration = pd.DataFrame({
        split: [metrics['calibration']['prediction_bias']]
        for split, metrics in results.items()
    }, index=['Prediction Bias'])
    
    calibration.plot(kind='bar', ax=axes[1,0])
    axes[1,0].set_title('Prediction Bias by Split Method')
    axes[1,0].set_ylabel('Bias')
    
    # 4. Performance Difference
    performance_diff = pd.DataFrame({
        split: [metrics['performance']['auc_difference']]
        for split, metrics in results.items()
    }, index=['AUC Difference'])
    
    performance_diff.plot(kind='bar', ax=axes[1,1])
    axes[1,1].set_title('AUC Difference (Train-Test) by Split Method')
    axes[1,1].set_ylabel('Difference')
    
    plt.tight_layout()
    plt.show()

# Display visualizations
plot_comprehensive_results(results)

,stratified,random,biased
Train Default Rate,0.073500,0.073200,0.078500
Test Default Rate,0.073500,0.074500,0.053500
Rate Difference,0.000000,0.001300,0.025000


,stratified,random,biased
Train AUC,1.000000,1.000000,1.000000
Test AUC,0.587300,0.594500,0.547800
AUC Difference,0.412700,0.405500,0.452200


,stratified,random,biased
Train Avg Prediction,0.074000,0.074200,0.079600
Test Avg Prediction,0.077000,0.075500,0.373300
Prediction Bias,0.003500,0.001000,0.319800


,stratified,random,biased
Chi-Square Statistic,0.000000,0.000000,0.182500
P-Value,1.000000,1.000000,0.669234
Is Significant,False,False,False


,stratified,random,biased
Expected Loss Impact (%),0.000000,0.125000,2.500000
Portfolio Risk Adjustment,0.000000,0.001200,0.024400
Threshold Adjustment Needed,0.000000,0.000600,0.012500


OSError: 'seaborn' is not a valid package style, path of style file, URL of style file, or library style name (library styles are listed in `style.available`)